# 🍏 Fun & Fit Health Advisor Agent Tutorial 🍎

Welcome to our **Fun & Fit Health Advisor Agent** tutorial, where you'll use **Microsoft Foundry** SDKs to create a playful (yet carefully disclaimed!) health and fitness assistant. We'll:

1. **Initialize** our project using **azure-ai-projects**.
2. **Create an Agent** specialized in providing general wellness and nutritional advice (with disclaimers!).
3. **Manage conversations** about fitness, nutrition, and general health topics.
4. **Showcase logging and tracing** with **OpenTelemetry**.
5. **Demonstrate** how to incorporate tools, safety disclaimers, and basic best practices.

### ⚠️ Important Medical Disclaimer ⚠️
> **The health information provided by this notebook is for general educational and entertainment purposes only and is not intended as a substitute for professional medical advice, diagnosis, or treatment.** Always seek the advice of your physician or other qualified health provider with any questions you may have regarding a medical condition. Never disregard professional medical advice or delay seeking it because of something you read or receive from this notebook.


## Prerequisites

Complete notebooks from the **Lab 00 - Prerequisite** section. 

## Let's Get Started
We'll walk you through each cell with notes and diagrams to keep it fun. Let's begin!

<img src="./seq-diagrams/1-basics.png" width="30%"/>




## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. 

* Open a terminal inside VSC (Visual Studio Code).
    * Run the following command in your terminal:

```
az login --use-device-code
```

* This will provide you with a device code and URL to authenticate in your browser to Azure.
    * Authenticate using the skillable Azure **Username** and **TAP**(Temporary Access Pass).
* Go back to the terminal and select the **default subscription.**

The Device Token will be used in this lab for:

* Remote development environments
* Systems without a default browser
* Corporate environments with strict security policies

* After successful authentication, you can proceed with the notebook cells below.

## 1. Initial Setup
We'll start by importing needed libraries, loading environment variables, and initializing an **AIProjectClient** so we can do all the agent-related actions. Let's do it! 🎉


In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

# Locate the repository .env file from the notebook working directory.
env_path = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError(
        "Could not find .env. Complete Lab 00 and place it in the repository root."
    )

load_dotenv(env_path)
tenant_id = os.environ.get("TENANT_ID")
ai_foundry_project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.environ.get("MODEL_DEPLOYMENT_NAME")
missing_variables = [
    name
    for name, value in {
        "TENANT_ID": tenant_id,
        "AI_FOUNDRY_PROJECT_ENDPOINT": ai_foundry_project_endpoint,
        "MODEL_DEPLOYMENT_NAME": model_deployment_name,
    }.items()
    if not value
]
if missing_variables:
    raise ValueError(f"Missing required .env variables: {', '.join(missing_variables)}")

print(f"📁 Environment loaded from: {env_path}")
print(f"🔑 Using Tenant ID: {tenant_id}")

# Uses the `az login --use-device-code` session created in the previous step.
credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(
    endpoint=ai_foundry_project_endpoint,
    credential=credential,
)
openai_client = project_client.get_openai_client()
print("✅ Successfully initialized AIProjectClient and OpenAI client")

## 2. Creating our Fun & Fit Health Advisor Agent 🏋️

We'll create an Agent specialized in general health and wellness. We'll explicitly mention disclaimers in its instructions, so it never forgets to keep it safe! The instructions also ask the agent to focus on general fitness, dietary tips, and always encourage the user to seek professional advice.


In [ ]:
def create_health_advisor_agent():
    """Create a versioned health advisor prompt agent."""
    agent = project_client.agents.create_version(
        agent_name="fun-fit-health-advisor",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="""
            You are a friendly AI Health Advisor.
            You provide general health, fitness, and nutrition information, but always:
            1. Include medical disclaimers.
            2. Encourage the user to consult healthcare professionals.
            3. Provide general, non-diagnostic advice around wellness, diet, and fitness.
            4. Clearly remind them you're not a doctor.
            5. Encourage safe and balanced approaches to exercise and nutrition.
            """,
        ),
    )
    print(f"🎉 Created agent {agent.name}, version: {agent.version}")
    return agent


health_advisor = create_health_advisor_agent()

## 3. Managing Our Health Conversation 💬

An OpenAI Conversation stores the user and assistant items for a multi-turn exchange. We will create one Conversation and reuse it for the health and fitness questions below.

In [ ]:
def start_health_conversation():
    """Create a Conversation for health and fitness questions."""
    conversation = openai_client.conversations.create()
    print(f"📝 Created a new Conversation, ID: {conversation.id}")
    return conversation


health_conversation = start_health_conversation()

## 4. Asking Health & Fitness Questions 🏃‍♂️
We'll create messages from the user about typical health questions. For example, **"How do I calculate my BMI?"** or **"What's a balanced meal for an active lifestyle?"**. We'll let our Health Advisor Agent respond, always remembering that disclaimer!


In [ ]:
health_responses = []


def chat_with_health_agent(user_question):
    """Send one turn to the health advisor in the shared Conversation."""
    print(f"👤 User: {user_question}")
    response = openai_client.responses.create(
        conversation=health_conversation.id,
        input=user_question,
        extra_body={
            "agent_reference": {
                "name": health_advisor.name,
                "type": "agent_reference",
            }
        },
    )
    health_responses.append(response)
    print(f"🤖 Agent: {response.output_text}")
    return response.output_text


print("🧪 Testing the Health & Fitness Agent...")
print("=" * 50)

### Example Queries
Let's do some quick queries now to see the agent's disclaimers and how it handles typical health questions. We'll ask about **BMI** and about **balanced meal** for an active lifestyle.


In [ ]:
questions = [
    "What is a balanced meal for an active lifestyle?",
    "What's a good 30-minute workout routine for a beginner who wants to build strength?",
    "What should I eat before and after a workout for optimal performance?",
    "How can I set realistic fitness goals for someone who wants to lose 20 pounds safely?",
]

responses = []
for index, question in enumerate(questions, start=1):
    print(f"\nTest {index}")
    print("-" * 30)
    responses.append(chat_with_health_agent(question))

successful_tests = sum(response is not None for response in responses)
print(f"\n📊 Summary: {successful_tests}/{len(questions)} responses received")

## 5. Final Test: Complex Health Question 🧹


In [ ]:
def show_conversation_history(conversation_id):
    """Display message items stored in an OpenAI Conversation."""
    items = openai_client.conversations.items.list(
        conversation_id=conversation_id,
        order="asc",
    )

    print(f"📋 Conversation History ({conversation_id})")
    print("=" * 60)
    message_count = 0

    for item in items:
        if item.type != "message":
            continue
        message_count += 1
        role = item.role.upper()
        text_parts = [
            block.text
            for block in item.content
            if block.type in {"input_text", "output_text"}
        ]
        print(f"\n{message_count}. {role}:")
        print("-" * 40)
        print("\n".join(text_parts))

    print(f"\n📊 Total messages in Conversation: {message_count}")


print("🔍 Final Test: Complex Health Question")
print("=" * 40)
final_response = chat_with_health_agent(
    "I'm a 35-year-old office worker who sits most of the day. I want to start "
    "exercising but have only 20 minutes, 3 times per week. What general exercise "
    "ideas could help, given that I also have back pain from sitting?"
)
show_conversation_history(health_conversation.id)

## 5. Cleanup 🧹
If you'd like to remove your agent from the service once finished, you can do so below. (In production, you might keep your agent around for stateful experiences!)

In [ ]:
openai_client.conversations.delete(conversation_id=health_conversation.id)
print("🗑️ Deleted Conversation")

project_client.agents.delete_version(
    agent_name=health_advisor.name,
    agent_version=health_advisor.version,
)
print("🗑️ Deleted agent version")

openai_client.close()
project_client.close()
credential.close()
print("✅ Cleanup completed!")

# Congratulations! 🏆
You've successfully built a **Fun & Fit Health Advisor** that can:
1. **Respond** to basic health and fitness questions.
2. **Use disclaimers** to encourage safe, professional consultation.
3. **Provide** general diet and wellness information.
4. **Use** the synergy of **Microsoft Foundry** services to power the conversation.

## Next Steps
- Explore adding more advanced tools (like **FileSearchTool** or **CodeInterpreterTool**) to provide more specialized info.
- Evaluate your AI's performance with **azure-ai-evaluation**!
- Add **OpenTelemetry** or Azure Monitor for deeper insights.
- Incorporate **function calling** if you want to handle things like advanced calculation or direct data analysis.

#### Let's proceed to [2-code_interpreter.ipynb](2-code_interpreter.ipynb)

Happy (healthy) coding! 💪

## 🎯 Summary & Next Steps

You created a versioned prompt agent with `PromptAgentDefinition`, invoked it through `openai_client.responses.create()`, and preserved multi-turn state with an OpenAI Conversation. The health disclaimer remains part of the agent instructions and each agent version is explicitly deleted during cleanup.